In [1]:
import numpy as np
import pandas as pd

import math
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from scipy.stats import chi2_contingency

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, auc, brier_score_loss
from sklearn.calibration import CalibrationDisplay
import torch
DEVICE = 'GPU' if torch.cuda.is_available() else 'CPU'

print(f"Using device: {DEVICE}")

Using device: GPU


In [2]:
df_external = pd.read_csv("external/f1_strategy_dataset_v4.csv")

In [3]:
df_external.head()

,Race,Driver,LapNumber,Stint,Compound,TyreLife,Normalized_TyreLife,LapTime_Sec,Position,GapToAhead_Sec,GapToBehind_Sec,TrackTemp_C,AirTemp_C,IsPitInLap,PitNextLap
0,Bahrain Grand Prix,VER,1,1,SOFT,1,0.0286,90.238,11,4.398,3.205,40.6,20.2,0,0
1,Bahrain Grand Prix,VER,2,1,SOFT,2,0.0571,90.516,2,3.749,4.723,30.0,29.9,0,0
2,Bahrain Grand Prix,VER,3,1,SOFT,3,0.0857,90.993,12,0.604,2.861,36.0,20.5,0,0
3,Bahrain Grand Prix,VER,4,1,SOFT,4,0.1143,91.092,16,1.547,0.908,39.3,23.8,0,0
4,Bahrain Grand Prix,VER,5,1,SOFT,5,0.1429,90.781,5,3.234,1.267,31.0,29.5,0,0


In [4]:
df_external[(df_external["Driver"]=="VER") & (df_external["PitNextLap"]==1)]

,Race,Driver,LapNumber,Stint,Compound,TyreLife,Normalized_TyreLife,LapTime_Sec,Position,GapToAhead_Sec,GapToBehind_Sec,TrackTemp_C,AirTemp_C,IsPitInLap,PitNextLap
16,Bahrain Grand Prix,VER,17,1,SOFT,17,0.4857,91.393,4,4.584,1.622,36.2,27.6,0,1
37,Bahrain Grand Prix,VER,38,2,HARD,20,0.5714,92.824,19,4.514,3.340,41.9,25.0,0,1
551,Saudi Arabian Grand Prix,VER,13,1,SOFT,13,0.3714,91.669,10,0.526,2.596,38.6,23.0,0,1
573,Saudi Arabian Grand Prix,VER,35,2,HARD,21,0.6000,92.841,4,3.533,0.782,33.2,28.1,0,1
1073,Australian Grand Prix,VER,14,1,SOFT,14,0.4000,90.929,13,0.757,3.528,38.0,26.6,0,1
1092,Australian Grand Prix,VER,33,2,HARD,18,0.5143,91.935,7,4.224,2.425,38.9,20.9,0,1
1613,Japanese Grand Prix,VER,14,1,MEDIUM,14,0.4000,92.751,13,1.802,1.641,35.0,22.7,0,1
1639,Japanese Grand Prix,VER,40,2,HARD,25,0.7143,93.915,15,1.675,2.804,34.8,21.7,0,1
2148,Chinese Grand Prix,VER,21,1,MEDIUM,21,0.6000,91.022,11,0.590,1.134,40.2,27.5,0,1


In [7]:
df_train = pd.read_csv("data/train.csv")
print(df_train.columns)
print(df_external.columns)
print(set(df_train.columns).difference(set(df_external.columns)))

Index(['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber',
       'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta',
       'Cumulative_Degradation', 'RaceProgress', 'Position_Change',
       'PitNextLap'],
      dtype='object')
Index(['Race', 'Driver', 'LapNumber', 'Stint', 'Compound', 'TyreLife',
       'Normalized_TyreLife', 'LapTime_Sec', 'Position', 'GapToAhead_Sec',
       'GapToBehind_Sec', 'TrackTemp_C', 'AirTemp_C', 'IsPitInLap',
       'PitNextLap'],
      dtype='object')
{'LapTime (s)', 'Year', 'PitStop', 'id', 'LapTime_Delta', 'Cumulative_Degradation', 'Position_Change', 'RaceProgress'}
